In [ ]:
import os
import cv2
import numpy as np
import time
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import tensorflow as tf
from tensorflow.keras.applications import ResNet50, ResNet152, ResNet101
from tensorflow.keras.applications.resnet import preprocess_input
from skimage import exposure
from skimage.io import imread
from skimage.transform import resize
from skimage.color import rgb2gray
import pandas as pd
from skimage.filters import gabor_kernel
from scipy import ndimage as ndi
from scipy.fftpack import dct
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D
import pandas as pd
import os
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, GlobalMaxPooling2D
import pandas as pd
import os

In [ ]:
import time
import tensorflow as tf

# ============================================================
# COMPUTATIONAL COMPLEXITY (TRI-STREAM EuPea-ResNet)
# ============================================================

# Literature-reported FLOPs @224×224
RESNET50_FLOPS  = 4.1e9     # 4.1 GFLOPs
RESNET101_FLOPS = 7.9e9     # 7.9 GFLOPs
RESNET152_FLOPS = 11.6e9    # 11.6 GFLOPs

RESNET50_PARAMS  = 25.6e6
RESNET101_PARAMS = 44.5e6
RESNET152_PARAMS = 60.2e6

total_flops = RESNET50_FLOPS + RESNET101_FLOPS + RESNET152_FLOPS
total_params = RESNET50_PARAMS + RESNET101_PARAMS + RESNET152_PARAMS

param_memory_mb = total_params * 4 / (1024**2)  # float32

print("\n========== Computational Complexity ==========")
print("Backbones used          : ResNet-50 + ResNet-101 + ResNet-152")
print(f"Total FLOPs (approx)    : {total_flops/1e9:.2f} GFLOPs")
print(f"Total Parameters        : {total_params/1e6:.2f} Million")
print(f"Parameter Memory        : {param_memory_mb:.2f} MB")

In [ ]:
import os
import tensorflow as tf
import pandas as pd
import numpy as np
from tensorflow.keras.layers import GlobalMaxPooling2D, GlobalAveragePooling2D, Concatenate, Input
from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50, ResNet101, ResNet152

# ------------------------------------------------------------
# 🔵 EuPea Attention Module
# ------------------------------------------------------------
class EuPea(tf.keras.layers.Layer):
    def __init__(self, epsilon=1e-5):
        super(EuPea, self).__init__()
        self.epsilon = epsilon

    def call(self, x):
        x_mean = tf.reduce_mean(x, axis=[1,2], keepdims=True)
        x_std  = tf.math.reduce_std(x, axis=[1,2], keepdims=True)

        d = tf.sqrt(tf.square(x - x_mean) + self.epsilon)
        d = (d - tf.reduce_min(d, axis=[1,2], keepdims=True)) / \
            (tf.reduce_max(d, axis=[1,2], keepdims=True) - tf.reduce_min(d, axis=[1,2], keepdims=True) + self.epsilon)

        z = tf.square((x - x_mean) / (x_std + self.epsilon))
        z = (z - tf.reduce_min(z, axis=[1,2], keepdims=True)) / \
            (tf.reduce_max(z, axis=[1,2], keepdims=True) - tf.reduce_min(z, axis=[1,2], keepdims=True) + self.epsilon)

        att = tf.nn.sigmoid(d + z)
        return x * att


In [ ]:
# ------------------------------------------------------------
# 🔵 Build EuPea-ResNet50 Feature Extractor
# ------------------------------------------------------------
def build_resnet50_eupea():
    base = ResNet50(weights="imagenet", include_top=False, input_shape=(224,224,3))
    inp = Input((224,224,3))
    x = tf.keras.applications.resnet50.preprocess_input(inp)
    att = EuPea()

    backbone = Model(inputs=base.input, outputs=[
        base.get_layer("pool1_pool").output,
        base.get_layer("conv2_block3_out").output,
        base.get_layer("conv3_block4_out").output,
        base.get_layer("conv4_block6_out").output,
        base.get_layer("conv5_block3_out").output
    ])

    feats = backbone(x)

    pooling = [
        GlobalMaxPooling2D(), GlobalMaxPooling2D(),
        GlobalAveragePooling2D(), GlobalAveragePooling2D(), GlobalAveragePooling2D()
    ]

    processed = [pool(att(f)) for f, pool in zip(feats, pooling)]
    return Model(inp, Concatenate()(processed))


# ------------------------------------------------------------
# 🔵 Build EuPea-ResNet101
# ------------------------------------------------------------
def build_resnet101_eupea():
    base = ResNet101(weights="imagenet", include_top=False, input_shape=(224,224,3))
    inp = Input((224,224,3))
    x = tf.keras.applications.resnet.preprocess_input(inp)
    att = EuPea()

    backbone = Model(inputs=base.input, outputs=[
        base.get_layer("pool1_pool").output,
        base.get_layer("conv2_block3_out").output,
        base.get_layer("conv3_block4_out").output,
        base.get_layer("conv4_block23_out").output,
        base.get_layer("conv5_block3_out").output
    ])

    feats = backbone(x)
    pooling = [
        GlobalMaxPooling2D(), GlobalMaxPooling2D(),
        GlobalAveragePooling2D(), GlobalAveragePooling2D(), GlobalAveragePooling2D()
    ]

    processed = [pool(att(f)) for f, pool in zip(feats, pooling)]
    return Model(inp, Concatenate()(processed))


# ------------------------------------------------------------
# 🔵 Build EuPea-ResNet152
# ------------------------------------------------------------
def build_resnet152_eupea():
    base = ResNet152(weights="imagenet", include_top=False, input_shape=(224,224,3))
    inp = Input((224,224,3))
    x = tf.keras.applications.resnet.preprocess_input(inp)
    att = EuPea()

    backbone = Model(inputs=base.input, outputs=[
        base.get_layer("pool1_pool").output,
        base.get_layer("conv2_block3_out").output,
        base.get_layer("conv3_block8_out").output,
        base.get_layer("conv4_block36_out").output,
        base.get_layer("conv5_block3_out").output
    ])

    feats = backbone(x)
    pooling = [
        GlobalMaxPooling2D(), GlobalMaxPooling2D(),
        GlobalAveragePooling2D(), GlobalAveragePooling2D(), GlobalAveragePooling2D()
    ]

    processed = [pool(att(f)) for f, pool in zip(feats, pooling)]
    return Model(inp, Concatenate()(processed))


In [ ]:
model_50  = build_resnet50_eupea()
model_101 = build_resnet101_eupea()
model_152 = build_resnet152_eupea()


In [ ]:
# ------------------------------------------------------------
# 🔵 Extract 50 + 101 + 152 + Concatenate + MinMax Normalize
# ------------------------------------------------------------
from sklearn.preprocessing import MinMaxScaler

labels = ['Type1', 'Type2', 'Type3']

def extract_tristream_features(data_dir):

    df = pd.DataFrame(columns=['features', 'labels', 'image_name'])

    for label_idx, label in enumerate(labels):
        folder = os.path.join(data_dir, label)
        if not os.path.exists(folder):
            print("⚠ Missing:", folder)
            continue

        for root, _, files in os.walk(folder):
            for file in files:
                if not file.lower().endswith((".jpg",".png",".jpeg")):
                    continue

                path = os.path.join(root, file)
                img = tf.keras.preprocessing.image.load_img(path, target_size=(224,224))
                x = tf.keras.preprocessing.image.img_to_array(img)[None, ...]

                f50  = model_50(x).numpy().flatten()
                f101 = model_101(x).numpy().flatten()
                f152 = model_152(x).numpy().flatten()

                fused = np.concatenate([f50, f101, f152])
                df.loc[len(df)] = [fused, label_idx, file]

    return df


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import label_binarize, MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    confusion_matrix, accuracy_score, f1_score, precision_score,
    precision_recall_curve, average_precision_score
)

# ============================================================
# PATHS & LABELS
# ============================================================
BASE_PATH = r"/kaggle/input/datasets/saurabhsainiji/iarc-trained-balanced-test-untouched/IARC-casewise-spilit-train-balanced-test-untouched/Fold-wise"
labels = ['Type - 1', 'Type - 2', 'Type - 3']

fold_results = {}
all_pr_curves = {i: {"precision": [], "recall": []} for i in range(len(labels))}

# ============================================================
# 5-FOLD TRAINING & EVALUATION
# ============================================================
for fold in range(1, 6):

    print(f"\n============== FOLD {fold} ==============")

    train_dir = os.path.join(BASE_PATH, f"Fold-{fold}", "Training")
    test_dir  = os.path.join(BASE_PATH, f"Fold-{fold}", "Testing")

    train_df = extract_tristream_features(train_dir)
    test_df  = extract_tristream_features(test_dir)

    X_train = np.stack(train_df['features'])
    y_train = np.array(train_df['labels'])

    X_test  = np.stack(test_df['features'])
    y_test  = np.array(test_df['labels'])

    # -----------------------------
    # Normalization
    # -----------------------------
    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    # -----------------------------
    # SVM with probability=True
    # FIX: removed CalibratedClassifierCV entirely
    # FIX: base_estimator → estimator (FutureWarning gone)
    # -----------------------------
    svm = SVC(kernel='linear', C=100, probability=True)
    svm.fit(X_train, y_train)

    y_pred = svm.predict(X_test)
    y_prob = svm.predict_proba(X_test)

    # -----------------------------
    # Metrics
    # -----------------------------
    cm = confusion_matrix(y_test, y_pred)
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro')
    f1   = f1_score(y_test, y_pred, average='macro')

    sensitivity, specificity = [], []
    for i in range(len(labels)):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP
        FP = cm[:, i].sum() - TP
        TN = cm.sum() - (TP + FP + FN)
        sensitivity.append(TP / (TP + FN + 1e-9))
        specificity.append(TN / (TN + FP + 1e-9))

    # -----------------------------
    # PR-AUC per class
    # -----------------------------
    y_true_bin = label_binarize(y_test, classes=[0, 1, 2])

    pr_auc = {}
    for i in range(len(labels)):
        precision_c, recall_c, _ = precision_recall_curve(
            y_true_bin[:, i], y_prob[:, i]
        )
        pr_auc[i] = average_precision_score(y_true_bin[:, i], y_prob[:, i])

        all_pr_curves[i]["precision"].append(precision_c)
        all_pr_curves[i]["recall"].append(recall_c)

        print(f"  {labels[i]}  PR-AUC = {pr_auc[i]:.4f}")

    pr_auc_macro = np.mean(list(pr_auc.values()))
    print(f"  Macro PR-AUC = {pr_auc_macro:.4f}")

    fold_results[f"Fold-{fold}"] = {
        "accuracy":           acc,
        "precision":          prec,
        "f1":                 f1,
        "sensitivity_macro":  np.mean(sensitivity),
        "specificity_macro":  np.mean(specificity),
        "pr_auc_per_class":   pr_auc,
        "pr_auc_macro":       pr_auc_macro
    }

# ============================================================
# 5-FOLD AVERAGES
# ============================================================
print("\n========== 5-Fold Average Metrics ==========")
# print(f"Accuracy     : {np.mean([v['accuracy']          for v in fold_results.values()]):.4f}")
# print(f"Precision    : {np.mean([v['precision']          for v in fold_results.values()]):.4f}")
# print(f"F1 Score     : {np.mean([v['f1']                 for v in fold_results.values()]):.4f}")
# print(f"Sensitivity  : {np.mean([v['sensitivity_macro']  for v in fold_results.values()]):.4f}")
# print(f"Specificity  : {np.mean([v['specificity_macro']  for v in fold_results.values()]):.4f}")
print(f"PR-AUC Macro : {np.mean([v['pr_auc_macro']       for v in fold_results.values()]):.4f}")

print("\n--- PR-AUC per class (5-fold avg) ---")
for i, lbl in enumerate(labels):
    avg = np.mean([fold_results[f"Fold-{f+1}"]["pr_auc_per_class"][i] for f in range(5)])
    print(f"  {lbl} : {avg:.4f}")

In [ ]:
import os
import numpy as np
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
import matplotlib.pyplot as plt
from sklearn.preprocessing import label_binarize, MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    confusion_matrix, accuracy_score, f1_score, precision_score,
    precision_recall_curve, average_precision_score
)

# ============================================================
# EXPECTED CALIBRATION ERROR FUNCTION
# ============================================================

def expected_calibration_error(y_true_bin, y_prob, n_bins=7):
    """
    Computes Expected Calibration Error (ECE) for multiclass (OvR)
    """
    ece_per_class = []

    for c in range(y_prob.shape[1]):
        prob_true, prob_pred = calibration_curve(
            y_true_bin[:, c], y_prob[:, c], n_bins=n_bins
        )

        bin_counts = np.histogram(
            y_prob[:, c], bins=n_bins, range=(0, 1)
        )[0]

        bin_weights = bin_counts / np.sum(bin_counts)

        L = min(len(prob_true), len(bin_weights))
        ece = np.sum(bin_weights[:L] * np.abs(prob_true[:L] - prob_pred[:L]))

        ece_per_class.append(ece)

    return ece_per_class, np.mean(ece_per_class)


# ============================================================
# PATHS & LABELS
# ============================================================

BASE_PATH = r"/kaggle/input/datasets/saurabhsainiji/iarc-trained-balanced-test-untouched/IARC-casewise-spilit-train-balanced-test-untouched/Fold-wise"
labels = ['Type - 1', 'Type - 2', 'Type - 3']

fold_results = {}
all_pr_curves = {i: {"precision": [], "recall": []} for i in range(len(labels))}

# For pooled calibration
pooled_y_true = []
pooled_y_prob = []

# ============================================================
# 5-FOLD TRAINING & EVALUATION
# ============================================================

for fold in range(1, 6):

    print(f"\n============== FOLD {fold} ==============")

    train_dir = os.path.join(BASE_PATH, f"Fold-{fold}", "Training")
    test_dir  = os.path.join(BASE_PATH, f"Fold-{fold}", "Testing")

    train_df = extract_tristream_features(train_dir)
    test_df  = extract_tristream_features(test_dir)

    X_train = np.stack(train_df['features'])
    y_train = np.array(train_df['labels'])

    X_test  = np.stack(test_df['features'])
    y_test  = np.array(test_df['labels'])

    # -----------------------------
    # Normalization
    # -----------------------------
    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    # -----------------------------
    # SVM + Isotonic Calibration
    # -----------------------------
    base_svm = SVC(kernel='linear', C=100, probability=True)

    svm = CalibratedClassifierCV(
        base_estimator=base_svm,
        method='isotonic',
        cv=3
    )

    svm.fit(X_train, y_train)

    y_pred = svm.predict(X_test)
    y_prob = svm.predict_proba(X_test)

    # Store pooled predictions
    pooled_y_true.append(y_test)
    pooled_y_prob.append(y_prob)

    # -----------------------------
    # Metrics
    # -----------------------------
    cm = confusion_matrix(y_test, y_pred)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')

    sensitivity = []
    specificity = []

    for i in range(len(labels)):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP
        FP = cm[:, i].sum() - TP
        TN = cm.sum() - (TP + FP + FN)

        sensitivity.append(TP / (TP + FN + 1e-9))
        specificity.append(TN / (TN + FP + 1e-9))

    sensitivity_macro = np.mean(sensitivity)
    specificity_macro = np.mean(specificity)

    # -----------------------------
    # PR-AUC
    # -----------------------------
    y_true_bin = label_binarize(y_test, classes=[0, 1, 2])

    pr_auc = {}
    precision_curve = {}
    recall_curve = {}

    for i in range(len(labels)):
        precision_curve[i], recall_curve[i], _ = precision_recall_curve(
            y_true_bin[:, i], y_prob[:, i]
        )
        pr_auc[i] = average_precision_score(y_true_bin[:, i], y_prob[:, i])

        all_pr_curves[i]["precision"].append(precision_curve[i])
        all_pr_curves[i]["recall"].append(recall_curve[i])

    fold_results[f"Fold-{fold}"] = {
        "accuracy": acc,
        "precision": prec,
        "f1": f1,
        "sensitivity_macro": sensitivity_macro,
        "specificity_macro": specificity_macro,
        "pr_auc_macro": np.mean(list(pr_auc.values()))
    }

# ============================================================
# 5-FOLD AVERAGES
# ============================================================

print("\n========== 5-Fold Average Metrics ==========")
# print("Accuracy     :", np.mean([v["accuracy"] for v in fold_results.values()]))
# print("Precision    :", np.mean([v["precision"] for v in fold_results.values()]))
# print("F1 Score     :", np.mean([v["f1"] for v in fold_results.values()]))
# print("Sensitivity  :", np.mean([v["sensitivity_macro"] for v in fold_results.values()]))
# print("Specificity  :", np.mean([v["specificity_macro"] for v in fold_results.values()]))
# print("PR-AUC       :", np.mean([v["pr_auc_macro"] for v in fold_results.values()]))

# ============================================================
# POOLED CALIBRATION CURVE
# ============================================================

pooled_y_true = np.concatenate(pooled_y_true)
pooled_y_prob = np.concatenate(pooled_y_prob, axis=0)

pooled_y_true_bin = label_binarize(pooled_y_true, classes=[0, 1, 2])

plt.figure(figsize=(8, 6))

for i in range(len(labels)):
    prob_true, prob_pred = calibration_curve(
        pooled_y_true_bin[:, i], pooled_y_prob[:, i], n_bins=7
    )
    plt.plot(prob_pred, prob_true, marker='o', linewidth=2, label=labels[i])

plt.plot([0, 1], [0, 1], 'k--', label="Perfect Calibration")
plt.xlabel("Predicted Probability", fontsize=16)
plt.ylabel("True Probability", fontsize=16)
plt.legend(fontsize=14)
plt.grid(True)
plt.show()

# ============================================================
# EXPECTED CALIBRATION ERROR (FINAL RESULT)
# ============================================================

ece_per_class, ece_macro = expected_calibration_error(
    pooled_y_true_bin,
    pooled_y_prob,
    n_bins=20
)

print("\n========== Expected Calibration Error (ECE) ==========")
for i, lbl in enumerate(labels):
    print(f"{lbl} ECE : {ece_per_class[i]:.4f}")

print(f"Macro ECE : {ece_macro:.4f}")